# P1 · Credit risk scoring: from raw data to the decision

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 110 minutes · **core path ★**

## Objective

Build on your own a model to decide whether to grant a loan, following the steps of lesson 10.1: closed test set, baseline, pipeline with dirty data (missing values, high-cardinality categories, an ordinal), comparison between a linear model and XGBoost, ablation of the missingness indicators, tuning, motivated choice, one-time check on the test set, threshold
decided with the **costs** (a missed defaulter costs 8 times a wrong refusal) and explanation.

## Dataset

`ds.loans(n=3000, seed=4)`: 8 columns (age, income, job tenure, amount, city, occupation with 40 levels, ordinal education, contract type) and `defaulted` (about 20%). There are missing values with different mechanisms: `income` is missing **more often among defaulters**, `job_tenure` is missing for the unemployed.

## Educational purpose

Put the whole course together in a single flow, respecting the rules you learned the hard way: no test data in the decisions, preprocessing only in the pipeline, comparison with the baseline and with the variability, choice guided by the standard error (the simplest model if the difference is in the noise), decision with the costs.

## Task

1. **Split:** `X_tr, X_te, y_tr, y_te` with `test_size=0.25`, `stratify=y`, `random_state=0`. From here on the test set is touched **only at step 7**.
2. **Preprocessing:** write `linear_prep()` (ColumnTransformer: numeric with median and **missingness indicator** then scaling; `education` with `OrdinalEncoder(categories=[ds.EDUCATION_ORDER], handle_unknown="use_encoded_value", unknown_value=-1)`, median with indicator, scaling; nominal `city`, `contract_type`, `occupation` with `OneHotEncoder(handle_unknown="ignore", min_frequency=15)`) and `tree_prep()` (numeric and `education` without imputation: XGBoost handles NaN; same one-hots).
3. **Comparison in CV (training set only):** `cv_table` (`pd.DataFrame`, index `baseline`, `logistic`, `xgb`; columns `auc`, `auc_sd`, `logloss`) with `cross_validate` on the 5 folds `cv` (given). Baseline: `DummyClassifier(strategy="prior")`; logistic: `LogisticRegression(max_iter=2000)`; xgb: `XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, eval_metric="logloss")`. All inside `make_pipeline`.
4. **Ablation:** `auc_without_indicators` (CV AUC of the logistic with `add_indicator=False` in both imputations) and `indicator_loss` (`cv_table.loc["logistic", "auc"] − auc_without_indicators`).
5. **Tuning:** `search = RandomizedSearchCV(xgb_pipeline, space, n_iter=12, scoring="neg_log_loss", cv=cv, random_state=0)` (`space` is given; the XGBoost step of `make_pipeline` is called `xgbclassifier`) and `tuned_logloss = −search.best_score_`.
6. **Choice:** `logistic_logloss_sd` (standard deviation of the log-loss across the 5 folds of the logistic) and `chosen` (`"xgb"` only if `tuned_logloss` is better than that of the logistic by **more** than `logistic_logloss_sd`; otherwise `"logistic"`). `chosen_model` is the corresponding pipeline (`search.best_estimator_` or the logistic).
7. **One-time check:** train `chosen_model` on the whole training set and save `test_auc` and `test_logloss` on the test set.
8. **Decision:** with `cross_val_predict(..., method="predict_proba")` on the training set choose `threshold` (grid `np.linspace(0.02, 0.9, 89)`) that minimizes `cost` (given). On the test set save `threshold_cost`, `cost_05` (threshold 0.5) and `refuse_all_cost` (refusing all the applicants).
9. **Explanation:** `top3` (list of the three original columns with the largest permutation importance on the **test set**, scoring `roc_auc`, `n_repeats=10`, `random_state=0`, in descending order).
10. **Report:** `report` (dictionary with `model`, `cv_auc`, `test_auc`, `threshold`, `threshold_cost`, `refuse_all_cost`, `top3` and `notes` (a string with two limits of the model that you would point out to whoever uses it)).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from scipy.stats import randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans(n=3000, seed=4)
y = df.pop("defaulted")
X = df
NUMERIC = ["age", "income", "job_tenure", "amount"]
NOMINAL = ["city", "contract_type", "occupation"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
space = {
    "xgbclassifier__max_depth": randint(2, 6), "xgbclassifier__learning_rate": [0.02, 0.05, 0.1], "xgbclassifier__n_estimators": [100, 200, 400],
    "xgbclassifier__min_child_weight": [1, 5, 10], "xgbclassifier__subsample": [0.7, 1.0], "xgbclassifier__colsample_bytree": [0.6, 1.0], "xgbclassifier__reg_lambda": [1, 5, 20],
}


def cost(y_true, p, threshold):
    """A wrong refusal (false positive) costs 1, an approved defaulter (false negative) costs 8."""
    y_true = np.asarray(y_true)
    return int(((p >= threshold) & (y_true == 0)).sum() * 1 + ((p < threshold) & (y_true == 1)).sum() * 8)

## Your work

### 1 · Split

In [ ]:
X_tr = X_te = y_tr = y_te = None

### 2 · Preprocessing

In [ ]:
def linear_prep(add_indicator=True):
    ...


def tree_prep():
    ...

### 3-4 · Comparison and ablation

In [ ]:
cv_table = pd.DataFrame(index=["baseline", "logistic", "xgb"], columns=["auc", "auc_sd", "logloss"], dtype=float)
auc_without_indicators = None
indicator_loss = None

### 5-6 · Tuning and choice

In [ ]:
search = None
tuned_logloss = None
logistic_logloss_sd = None
chosen = None
chosen_model = None

### 7-8 · One-time check and decision

In [ ]:
test_auc = test_logloss = None
threshold = threshold_cost = cost_05 = refuse_all_cost = None

### 9 · Explanation and report

In [ ]:
top3 = []
report = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Linear preprocessing: `ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=ind), StandardScaler()), NUMERIC), ("edu", make_pipeline(OrdinalEncoder(...), SimpleImputer(strategy="median", add_indicator=ind), StandardScaler()), ["education"]), ("nom", OneHotEncoder(...), NOMINAL)])`. For `cv_table`: `r = cross_validate(model, X_tr, y_tr, cv=cv, scoring=["roc_auc", "neg_log_loss"])`, `r["test_roc_auc"].mean()`, `.std()`, `-r["test_neg_log_loss"].mean()`. For `top3`: `pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False).index[:3].tolist()`.'):
    check(len(X_tr) == 2250 and len(X_te) == 750 and abs(y_tr.mean() - y_te.mean()) < 0.005, "stratified 75/25 split (2250 and 750 rows, same prevalence)", "train_test_split(..., stratify=y)")
    Z = linear_prep().fit_transform(X_tr)
    Z = Z.toarray() if hasattr(Z, "toarray") else np.asarray(Z)
    A = tree_prep().fit_transform(X_tr)
    A = A.toarray() if hasattr(A, "toarray") else np.asarray(A)
    check(Z.shape[0] == 2250 and not np.isnan(Z.astype(float)).any() and A.shape[0] == 2250 and np.isnan(A.astype(float)).any() and Z.shape[1] > A.shape[1] - 5, "linear_prep leaves no NaN; tree_prep leaves them (for XGBoost)", "SimpleImputer only in the linear version")
    check(cv_table.notna().all().all() and abs(cv_table.loc["baseline", "auc"] - 0.5) < 0.02 and cv_table.loc["logistic", "auc"] > 0.8 and cv_table.loc["xgb", "auc"] > 0.79 and abs(cv_table.loc["logistic", "auc"] - cv_table.loc["xgb", "auc"]) < 0.03, "baseline at 0.5, logistic above 0.80, xgb above 0.79 and the two models within 0.03 of each other", "cross_validate with scoring=['roc_auc', 'neg_log_loss']")
    check(cv_table.loc["baseline", "logloss"] > cv_table.loc["logistic", "logloss"] + 0.1 and (cv_table.loc[["logistic", "xgb"], "auc_sd"] > 0).all(), "the baseline log-loss exceeds that of the logistic by over 0.1; positive standard deviations for the two models", "-test_neg_log_loss")
    check(auc_without_indicators is not None and indicator_loss > 0.02, "without the missingness indicators the CV AUC drops by over 0.02: the missingness carries information", "SimpleImputer(add_indicator=False)")
    check(search is not None and len(search.cv_results_["params"]) == 12 and tuned_logloss < 0.375, "12 configurations tried; log-loss of the best below 0.375", "RandomizedSearchCV(..., n_iter=12)")
    check(logistic_logloss_sd is not None and 0.005 < logistic_logloss_sd < 0.05 and chosen == "logistic", "the difference between tuned xgb and logistic is smaller than the standard deviation across folds: the logistic is chosen", "logloss(logistic) - tuned_logloss > sd?")
    check(test_auc is not None and 0.77 < test_auc < 0.86 and abs(test_auc - cv_table.loc["logistic", "auc"]) < 0.05, "test AUC between 0.77 and 0.86 and consistent with the CV one (within 0.05)", "chosen_model.fit(X_tr, y_tr) and then evaluate on the test set")
    check(threshold is not None and 0.05 <= threshold <= 0.4 and threshold_cost < 0.75 * cost_05 and threshold_cost < refuse_all_cost, "threshold between 0.05 and 0.4; cost on the test set at least 25% below that of threshold 0.5 and below refusing everyone", "argmin of cost(y_tr, p_cv, s)")
    check(len(top3) == 3 and top3[0] == "income" and "amount" in top3, "`income` is the most important feature and `amount` is among the top three", "permutation_importance on X_te, y_te")
    check(set(report) >= {"model", "cv_auc", "test_auc", "threshold", "threshold_cost", "refuse_all_cost", "top3", "notes"} and isinstance(report["notes"], str) and len(report["notes"]) > 40, "complete report, with two limits in the notes", "a dictionary with the required keys")
done()

---
The commented solution is in `solutions/B_with_sklearn/P1_credit_risk_scoring_from_data_to_decision.ipynb`. Open it after you pass the check, or after you have really tried.